## Project Description: Next Word Prediction Using LSTM
#### Project Overview:

This project aims to develop a deep learning model for predicting the next word in a given sequence of words. The model is built using Long Short-Term Memory (LSTM) networks, which are well-suited for sequence prediction tasks. The project includes the following steps:

1- Data Collection: We use the text of Shakespeare's "Hamlet" as our dataset. This rich, complex text provides a good challenge for our model.

2- Data Preprocessing: The text data is tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into training and testing sets.

3- Model Building: An LSTM model is constructed with an embedding layer, two LSTM layers, and a dense output layer with a softmax activation function to predict the probability of the next word.

4- Model Training: The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when the loss stops improving.

5- Model Evaluation: The model is evaluated using a set of example sentences to test its ability to predict the next word accurately.

6- Deployment: A Streamlit web application is developed to allow users to input a sequence of words and get the predicted next word in real-time.

In [1]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd

[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/aryanfursule/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [2]:
data = gutenberg.raw('shakespeare-hamlet.txt')

with open('hamlet.txt', 'w') as file:
    file.write(data)

In [3]:
import tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

with open('hamlet.txt', 'r') as file:
    text = file.read().lower()

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index)

In [4]:
input_seq=[]
for line in text. split('\n'):
    token_list=tokenizer.texts_to_sequences([line])[0]
    for i in range (1, len (token_list)) :
        n_gram_sequence=token_list[: i+1]
        input_seq.append(n_gram_sequence)

In [5]:
import numpy as np
max_seq_len = max([len(x) for x in input_seq])

input_seq = np.array(pad_sequences(input_seq, maxlen = max_seq_len, padding = 'pre'))
input_seq

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]], dtype=int32)

In [16]:
X, y = input_seq[:,:-1], input_seq[:, -1]

In [17]:
import tensorflow as tf
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## LSTM Training:

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [29]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM, Dropout

model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_seq_len-1))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])
model.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_5 (Embedding)     (None, 13, 100)           481700    
                                                                 
 lstm_7 (LSTM)               (None, 13, 150)           150600    
                                                                 
 dropout_3 (Dropout)         (None, 13, 150)           0         
                                                                 
 lstm_8 (LSTM)               (None, 100)               100400    
                                                                 
 dense_2 (Dense)             (None, 4817)              486517    
                                                                 
Total params: 1219217 (4.65 MB)
Trainable params: 1219217 (4.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [30]:
history = model.fit(X_train, y_train, epochs=50, validation_data=(X_test, y_test), verbose=1)

Epoch 1/50


2026-06-22 17:39:51.769483: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


644/644 [==============================] - 15s 17ms/step - loss: 6.8999 - accuracy: 0.0327 - val_loss: 6.7555 - val_accuracy: 0.0326
Epoch 2/50
644/644 [==============================] - 9s 14ms/step - loss: 6.4683 - accuracy: 0.0387 - val_loss: 6.8732 - val_accuracy: 0.0437
Epoch 3/50
644/644 [==============================] - 9s 14ms/step - loss: 6.3265 - accuracy: 0.0446 - val_loss: 6.8701 - val_accuracy: 0.0476
Epoch 4/50
644/644 [==============================] - 9s 15ms/step - loss: 6.1730 - accuracy: 0.0529 - val_loss: 6.8869 - val_accuracy: 0.0530
Epoch 5/50
644/644 [==============================] - 9s 14ms/step - loss: 6.0214 - accuracy: 0.0586 - val_loss: 6.9235 - val_accuracy: 0.0567
Epoch 6/50
644/644 [==============================] - 10s 15ms/step - loss: 5.8715 - accuracy: 0.0668 - val_loss: 6.9971 - val_accuracy: 0.0600
Epoch 7/50
644/644 [==============================] - 9s 15ms/step - loss: 5.7181 - accuracy: 0.0731 - val_loss: 7.0182 - val_accuracy: 0.0653
Epoch 8/

In [ ]:
def predict_next_word (model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len (token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):] # Ensure the sequence length matches max_sequ
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model. predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)
    for word, index in tokenizer. word_index.items():  
        if index == predicted_word_index:
            return word
    return None

In [49]:
input_text=" Barn. Last night of all, When yond same" 
print (f"Input text: {input_text}")
max_sequence_len=model.input_shape[1]+1
next_word=predict_next_word (model, tokenizer, input_text, max_sequence_len)
print (f"Next Word Prediction: {next_word}")

Input text:  Barn. Last night of all, When yond same
Next Word Prediction: starre


In [45]:
import pickle as pkl
model.save("next_word_lstm.h5")

with open('tokenizer.pkl', 'wb') as handle:
    pkl.dump(tokenizer, handle, protocol=pkl.HIGHEST_PROTOCOL)

/Users/aryanfursule/Desktop/DeepLearning/venv/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
